### Introduction to MPI in Python

##### What is it? Why do we Care?
MPI stands for Message Passing Interface and is used for running multiple different processes that work together on a task to speed up task completion.

Sometimes, you'll hear the term thread and process used together when discussing parallel computations. Threads exist within processes, and they share an address space in memory. That means that if we have an array `a` in thread 1, thread 2 can also access all of the underlying data in `a` instead of receiving a copy. A process is a running program instance with its own isolated memory space (this is why Chrome on your computer cannot access, for example, the weather app). Since processes have isolated memory, they need to communicate with each other through message passing.

Practically, this means that processes include more overhead and are slower to set up and communicate/switch between. For example, in C++ we would be able to access the same underlying array across threads. However, in Python, we are somewhat limited to parallelizing with processes due to something called the Global Interpreter Lock (GIL). Python is an interpreted scripting language and one of its features is that while multiple threads can exist, only one can run at a time. This limits us to a single CPU if we use multiple threads. So in Python we typically spin up multiple processes instead and communicate across them instead (hence usage of MPI).

Here is a diagram for visualization:

mpiexec -n 4 python program.py

                  same program.py
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓            ↓
      process 0    process 1    process 2    process 3
       rank=0       rank=1       rank=2       rank=3

          │            │            │            │
          └──── communicate using MPI ───────────┘

##### Setup
Run from project root: `uv sync`.

This creates `.venv` and installs `mpi4py`, the Python package we use to talk to MPI. `mpi4py` is only a wrapper, though. It needs an actual MPI library on your computer, plus the `mpiexec` launcher that starts the processes.

On macOS and Linux there shouldn't be anything additional to do.

On Windows, you may have to install Microsoft MPI in PowerShell:

```powershell
winget install Microsoft.msmpi
```

Finally, pick `.venv` as the kernel for this notebook (in VS Code: Select Kernel, then Python Environments, then `.venv`). Try running the cell below to see if it works.

In [ ]:
%%writefile notebook_scripts/test_mpi.py

from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

print(f"Rank: {rank}")

In [ ]:
!uv run mpiexec -n 4 python notebook_scripts/test_mpi.py

##### Demo
Now let's see how it speeds up the task when we have a large array of computations to perform.

In [ ]:
%%writefile notebook_scripts/demo.py

# Suppose that we want to find sum(x^2 for x in range(0, 20,000,000))
# We compare the time it takes to parallelize this with 4 processes vs. 1

from mpi4py import MPI

comm = MPI.COMM_WORLD
rank = comm.Get_rank() # the identity of the process within world
world_size = comm.Get_size() # number of processes total

N = 20_000_000
chunk_size = N // world_size

start_idx = rank * chunk_size
end_idx = N if rank == world_size - 1 else start_idx + chunk_size # exclusive end

comm.Barrier() # pauses every process until each one reaches this point in code (since process startup order is random and they may finish at different paces)

start_time = MPI.Wtime()

rank_local_sum = 0
for i in range(start_idx, end_idx):
    rank_local_sum += i ** 2

global_sum = comm.reduce(rank_local_sum, op=MPI.SUM, root=0) # take sum of each rank-local sum and give it to rank=0 process
end_time = MPI.Wtime()
time_spent = end_time - start_time

if rank == 0:
    print(f"Result: {global_sum}")
    print(f"Time Spent: {time_spent:.3f} seconds")

In [ ]:
!uv run mpiexec -n 4 python notebook_scripts/demo.py

In [ ]:
# Compare the above result to this

import time

N = 20_000_000
total = 0
start_time = time.perf_counter()
for i in range(N):
    total += i ** 2
end_time = time.perf_counter()
time_spent = end_time - start_time

print(f"Result: {total}")
print(f"Time Spent: {time_spent:.3f} seconds")

Below is an image of runtime on my laptop:

<img src="images/2026-09-19-17-03-20.png" style="width: 500px; max-width: 100%; height: auto;">

##### Practice
Write an MPI program that counts how many prime numbers exist from 2 through N.

Example:
```
N = 10
primes = 2, 3, 5, 7
answer = 4
```

Write one version that is sequential and one that utilizes parallelization.

In [ ]:
from math import sqrt
import time

start_time = time.perf_counter()

def is_prime(num: int) -> bool:
    if num == 2:
        return True
    
    limit = int(sqrt(num))
    for divisor in range(2, limit + 1):
        if num % divisor == 0:
            return False

    return True

def count_primes(n: int) -> int:
    if n < 2:
        return 0

    counter = 0
    for num in range(2, n + 1):
        if is_prime(num):
            counter += 1

    return counter

print(count_primes(1_000_000))

end_time = time.perf_counter()
print(f"Time taken: {(end_time - start_time):.3f}s")

In [ ]:
%%writefile notebook_scripts/count_primes.py


from math import sqrt
from tracemalloc import start
from mpi4py import MPI

start_time = MPI.Wtime()

def is_prime(num: int) -> bool:
    if num == 2:
        return True

    limit = int(sqrt(num))
    for divisor in range(2, limit + 1):
        if num % divisor == 0:
            return False

    return True

def count_primes(n: int, comm: MPI.Comm) -> int:
    if n < 2:
        return 0

    world_size = comm.Get_size()
    rank = comm.Get_rank()
    chunk_size = n // world_size
    count = 0
    start_num = max(2, chunk_size * rank)
    end_num = n if rank == world_size - 1 else start_num + chunk_size

    for num in range(start_num, end_num + 1):
        if is_prime(num):
            count += 1

    return count

comm = MPI.COMM_WORLD
rank_local_count = count_primes(1_000_000, comm)
global_count = comm.reduce(rank_local_count, MPI.SUM, root=0) # None for the non-root ranks

end_time = MPI.Wtime()
rank_local_time = end_time - start_time
total_time_taken = comm.reduce(rank_local_time, op=MPI.MAX, root=0)

if comm.Get_rank() == 0:
    print(f"Count: {global_count}")
    print(f"Time: {total_time_taken:3f}")

In [ ]:
!uv run mpiexec -n 4 python notebook_scripts/count_primes.py